# Part 2: Python OOP - Senior Level

Object-oriented Python is about designing objects that combine state, behavior, and clear contracts. This notebook moves from the object model to advanced language mechanics and design principles.

## Learning goals

By the end of this notebook, you should be able to:

- Design classes with deliberate ownership, state, and public interfaces.
- Choose composition, inheritance, abstract base classes, or protocols appropriately.
- Explain method binding, MRO, `super()`, object construction, descriptors, and `__slots__`.
- Recognize the risks of finalizers and metaclasses.
- Apply SOLID, DRY, KISS, dependency inversion, and dependency injection in Python code.

> Senior-level OOP is not about using more classes. It is about making responsibilities, substitutions, dependencies, and invariants explicit.

## 1. Core OOP

### Classes and objects

A class is a blueprint and a namespace for behavior. An object is an instance created from that class. `self` is the instance passed to an instance method; it is not a keyword, but the conventional name.

### Instance state versus class state

- **Instance variables** belong to one object and are normally created with `self.name`.
- **Class variables** live on the class and are shared through lookup by instances until an instance assigns an attribute with the same name.

Avoid mutable class variables for per-instance state. Use a class variable for shared configuration or counters only when sharing is intentional.

### Method kinds

- An **instance method** receives `self` and can read or mutate instance state.
- A **class method** receives `cls` and is useful for alternate constructors or class-level behavior.
- A **static method** receives no automatic instance or class reference; it is a namespaced utility that logically belongs with the class.

### Encapsulation and properties

Python uses conventions rather than enforced private fields: `_name` means internal use, while `__name` triggers name mangling. Encapsulation means protecting invariants behind a stable interface, not merely hiding data. `@property` lets attribute-like access run validation or computed behavior while preserving a clean API.

### Composition, inheritance, and polymorphism

**Composition** gives an object other objects to delegate to: a `Car` has an `Engine`. **Inheritance** expresses an is-a relationship and should preserve substitutability. Prefer composition when behavior changes independently or when reuse does not represent a true subtype.

**Polymorphism** means client code can use different objects through the same operation. Python commonly uses duck typing: if an object supports the required behavior, its concrete class may not matter. Inheritance, ABCs, and protocols can make that contract more explicit.

### Abstract classes and interfaces

`abc.ABC` and `@abstractmethod` define runtime-enforced abstract contracts. `typing.Protocol` defines a structural interface: a class can satisfy it without inheriting from it, provided it has the required members. Protocols are often a strong fit for dependency inversion because they describe what a collaborator must do rather than what it must inherit from.

In [4]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

class Engine:
    def start(self):
        return "engine started"

class Vehicle(ABC):
    wheels = 4  # Intentional class variable shared by the type.

    def __init__(self, name, engine):
        self.name = name  # Instance variable.
        self.engine = engine  # Composition: Vehicle has an Engine.
        self._speed = 0

    def accelerate(self, amount=10):
        self._speed += amount
        return self._speed

    @classmethod
    def named(cls, name, engine):
        return cls(name, engine)  # Alternate constructor.

    @staticmethod
    def valid_speed(speed):
        return speed >= 0

    @property
    def speed(self):
        return self._speed

    @speed.setter
    def speed(self, value):
        if not self.valid_speed(value):
            raise ValueError("speed cannot be negative")
        self._speed = value

    @abstractmethod
    def move(self):
        """Return a description of movement."""

class Car(Vehicle):
    def move(self):
        return f"{self.name} drives"

class Bicycle:
    wheels = 2

    def move(self):
        return "bicycle pedals"

@runtime_checkable
class Movable(Protocol):
    def move(self) -> str:
        ...

def report_movement(vehicle: Movable):
    return vehicle.move()

car = Car.named("Comet", Engine())
print("instance state:", car.name, car.engine.start())
print("class state:", car.wheels, Car.wheels)
print("methods:", car.accelerate(20), Vehicle.valid_speed(car.speed))
car.speed = 55
print("validated property:", car.speed)
print("polymorphism:", report_movement(car), report_movement(Bicycle()))

try:
    Vehicle("abstract", Engine())
except TypeError as error:
    print("ABC prevents incomplete instance:", error)

# Protocols are structural: Bicycle does not inherit from Movable.
print("protocol-style compatibility:", isinstance(Bicycle(), Movable))

instance state: Comet engine started
class state: 4 4
methods: 20 True
validated property: 55
polymorphism: Comet drives bicycle pedals
ABC prevents incomplete instance: Can't instantiate abstract class Vehicle without an implementation for abstract method 'move'
protocol-style compatibility: True


## 2. Advanced OOP mechanics

### MRO, multiple inheritance, and `super()`

Python calculates a class's **method resolution order** (MRO) using C3 linearization. It determines where attribute and method lookup proceeds. Inspect it with `Class.__mro__` or `Class.mro()`.

Multiple inheritance is safest when mixins are small and cooperative. A cooperative method calls `super()` so the next class in the MRO can participate. `super()` does not simply mean "call my parent"; it means "continue lookup after this class in the current MRO". This is why cooperative multiple inheritance can work without hard-coding parent names.

### `__new__`, `__init__`, and `__del__`

`__new__` creates and returns an instance, so it matters for immutable types, singletons, and metaclass-controlled creation. `__init__` initializes an already-created instance and must return `None`; it does not create the object.

`__del__` is a finalizer with weak guarantees. It may run late, not run at interpreter shutdown, and be delayed by reference cycles. Do not use it for critical resource release. Prefer context managers (`with`) and explicit `close` methods.

### Dunder methods

Magic methods define how objects integrate with Python syntax: `__repr__`, `__str__`, `__len__`, `__iter__`, `__contains__`, `__getitem__`, `__eq__`, `__hash__`, arithmetic methods, and context-manager methods such as `__enter__` and `__exit__`. Implement only operations that make semantic sense and preserve Python's expectations.

### Descriptors and properties

A descriptor is an object defining `__get__`, `__set__`, or `__delete__`. Descriptors control attribute access. Functions become bound methods through the descriptor protocol, and `property` is a built-in descriptor. Custom descriptors are useful for reusable validation, conversion, logging, or lazy attributes.

### `__slots__`

`__slots__` declares permitted instance attributes and can remove the per-instance `__dict__`, reducing memory for many small objects. It also prevents arbitrary new attributes and can complicate inheritance, weak references, and pickling. Use it when memory or attribute constraints justify the tradeoff, not automatically.

### Metaclasses

A metaclass is the class of a class; normally it is `type`. It can customize class creation, registration, or validation. Metaclasses affect an entire class hierarchy and add complexity, so prefer class decorators, `__init_subclass__`, descriptors, or ordinary functions when they solve the problem.

In [5]:
class LoggedMixin:
    def save(self):
        events = super().save()
        return events + [f"logged:{self.name}"]

class Persisted:
    def save(self):
        return [f"saved:{self.name}"]

class Document(LoggedMixin, Persisted):
    def __init__(self, name):
        self.name = name

print("MRO:", [class_type.__name__ for class_type in Document.__mro__])
print("cooperative super:", Document("notes").save())

class Positive:
    def __set_name__(self, owner, name):
        self.private_name = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.private_name)

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError("value must be positive")
        setattr(instance, self.private_name, value)

class Measurement:
    amount = Positive()

    def __init__(self, amount):
        self.amount = amount

measurement = Measurement(3)
measurement.amount = 5
print("descriptor validation:", measurement.amount)
try:
    measurement.amount = 0
except ValueError as error:
    print("descriptor rejected value:", error)

class CompactUser:
    __slots__ = ("name",)

    def __init__(self, name):
        self.name = name

compact_user = CompactUser("Ada")
print("slots value:", compact_user.name)
try:
    compact_user.email = "ada@example.com"
except AttributeError as error:
    print("slots reject undeclared attributes:", error)

class Token:
    def __new__(cls, text):
        instance = super().__new__(cls)
        instance.text = text.upper()
        return instance

    def __init__(self, text):
        self.original = text

    def __repr__(self):
        return f"Token(text={self.text!r}, original={self.original!r})"

print("construction order:", Token("py"))
print("metaclass of Token:", type(Token).__name__)

class Registered:
    registry = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Registered.registry.append(cls.__name__)

class Plugin(Registered):
    pass

print("class registration without custom metaclass:", Registered.registry)

MRO: ['Document', 'LoggedMixin', 'Persisted', 'object']
cooperative super: ['saved:notes', 'logged:notes']
descriptor validation: 5
descriptor rejected value: value must be positive
slots value: Ada
slots reject undeclared attributes: 'CompactUser' object has no attribute 'email' and no __dict__ for setting new attributes
construction order: Token(text='PY', original='py')
metaclass of Token: type
class registration without custom metaclass: ['Plugin']


## 3. Design principles

### SOLID

- **Single Responsibility:** a class should have one reason to change. Separate persistence, formatting, and business rules when they evolve independently.
- **Open/Closed:** extend behavior through composition, polymorphism, or plugins instead of repeatedly editing stable code.
- **Liskov Substitution:** a subtype must honor the expectations of its base type. Do not weaken guarantees, surprise callers, or make valid base operations fail.
- **Interface Segregation:** prefer small focused protocols over one large interface that forces unused methods on clients.
- **Dependency Inversion:** high-level policy should depend on abstractions, not concrete infrastructure details.

### DRY and KISS

**DRY** means one authoritative source for knowledge that must stay consistent. It does not mean eliminating every similar-looking line; premature abstraction can make code harder to understand. **KISS** favors the simplest design that meets the requirements. A plain function or dataclass is often better than a hierarchy.

### Composition over inheritance

Use inheritance for a stable subtype relationship and substitutable behavior. Use composition when you want to assemble capabilities, swap collaborators, or avoid a rigid hierarchy. Mixins should provide narrow, cohesive behavior and cooperate with `super()` when combined.

### Dependency inversion and dependency injection

A service should express what it needs through a protocol or small callable, while the application boundary supplies the concrete implementation. This is **dependency injection**. Constructor injection is usually the clearest form because required dependencies are explicit and easy to replace in tests.

The following example keeps business logic independent from a concrete storage system. Both the real adapter and the test double satisfy the same protocol by behavior.

In [6]:
from typing import Protocol

class UserStore(Protocol):
    def save(self, username: str) -> None:
        ...

class InMemoryUserStore:
    def __init__(self):
        self.users = []

    def save(self, username: str) -> None:
        self.users.append(username)

class UserService:
    def __init__(self, store: UserStore):
        self.store = store  # Constructor injection.

    def register(self, username: str) -> str:
        if not username.strip():
            raise ValueError("username is required")
        self.store.save(username)
        return f"registered:{username}"

store = InMemoryUserStore()
service = UserService(store)
print("production-style dependency injection:", service.register("Ada"), store.users)

# A tiny test double makes the high-level service easy to test.
class RecordingStore:
    def __init__(self):
        self.saved = []

    def save(self, username: str) -> None:
        self.saved.append(username)

fake_store = RecordingStore()
print("test double:", UserService(fake_store).register("Grace"), fake_store.saved)

# Composition lets behavior vary without creating a subclass hierarchy.
class EmailNotifier:
    def send(self, message):
        return f"email:{message}"

class AuditNotifier:
    def send(self, message):
        return f"audit:{message}"

class NotifyingService:
    def __init__(self, notifier):
        self.notifier = notifier

    def announce(self, message):
        return self.notifier.send(message)

print("composed behavior:", NotifyingService(EmailNotifier()).announce("welcome"))
print("swapped behavior:", NotifyingService(AuditNotifier()).announce("welcome"))

production-style dependency injection: registered:Ada ['Ada']
test double: registered:Grace ['Grace']
composed behavior: email:welcome
swapped behavior: audit:welcome


## 4. Senior-level practice lab

Attempt each problem before running your own solution.

1. Create a `BankAccount` class with a private balance, a property that rejects negative balances, and `deposit`/`withdraw` methods that preserve the invariant.
2. Add an alternate constructor `from_string` using `@classmethod`; keep validation in one place.
3. Create a `Shape` ABC with an abstract `area` method. Implement `Rectangle` and `Circle`, then calculate areas polymorphically.
4. Define a `SupportsWrite` protocol and write a function that accepts any object with a compatible `write(text)` method.
5. Build two cooperative mixins that each contribute to a `describe()` method. Print the class MRO and explain the output order.
6. Implement a reusable descriptor that validates a field is a non-empty string. Use it on two different classes.
7. Compare an ordinary class with an equivalent `__slots__` class using `hasattr(instance, "__dict__")`. Explain the memory and flexibility tradeoff.
8. Write a context manager class with `__enter__` and `__exit__` that records whether its block succeeded or raised an exception.
9. Refactor a service that directly constructs a concrete repository so the repository is injected through a small protocol.
10. Review one of your solutions against SOLID, DRY, KISS, and composition-over-inheritance. Record one justified tradeoff.

### Review checklist

- [ ] Does each class have one clear responsibility?
- [ ] Are mutable values stored per instance rather than accidentally on the class?
- [ ] Does inheritance represent a valid is-a relationship?
- [ ] Could a protocol or callable express the dependency more clearly?
- [ ] Does `super()` participate correctly in multiple inheritance?
- [ ] Are invariants protected by methods or properties?
- [ ] Is `__del__` avoided for critical cleanup?
- [ ] Is advanced machinery justified by a concrete requirement?
- [ ] Can collaborators be replaced easily in a test?

### Final perspective

Python gives you many extension points, but senior design is selective. Start with a function or dataclass, add composition and dependency injection when variation appears, use protocols for focused contracts, and reach for inheritance, descriptors, slots, or metaclasses only when their specific semantics solve a real problem.